# scikit-learn: A Hands-On Lesson

scikit-learn is the standard Python library for classical machine learning. It gives you one consistent interface for preprocessing, models, evaluation, and tuning.

Every scikit-learn object follows the same pattern:

- **`fit(X, y)`** learns from training data.
- **`predict(X)`** makes predictions.
- **`transform(X)`** changes the data (for preprocessing tools).
- **`score(X, y)`** returns a default quality measure.

This lesson follows a realistic workflow for a **cost prediction** problem:

1. Make a dataset (synthetic, so no download is needed)
2. Split into training and test sets
3. Preprocess features with pipelines
4. Start with a baseline
5. Compare linear regression and gradient boosting
6. Evaluate with proper metrics and cross-validation
7. Understand which features matter
8. Practice exercises, with solutions

The data is simulated, so the relationships we find are the ones we built in. The workflow is what matters.

## 1. Setup and synthetic data

On some systems (including WSL2), scikit-learn's gradient boosting can hang when it uses many threads at once. Limiting it to one thread avoids the problem. This line must run before scikit-learn is imported, so it goes first.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"

Next we build the dataset, using a seeded random generator so the results repeat:

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=0)
n = 2000

age = rng.integers(18, 85, size=n)
bmi = rng.normal(27, 5, size=n).clip(15, 50)
smoker = rng.choice(["yes", "no"], size=n, p=[0.2, 0.8])
region = rng.choice(["north", "south", "east", "west"], size=n)
chronic_conditions = rng.poisson(0.8, size=n)

# Costs depend on the features, with a skewed, noisy tail like real claims data
base = (
    500
    + 40 * age
    + 60 * np.maximum(bmi - 25, 0)
    + 6000 * chronic_conditions
    + 8000 * (smoker == "yes")
)
cost = base * rng.lognormal(mean=0, sigma=0.25, size=n)

df = pd.DataFrame({
    "age": age,
    "bmi": bmi.round(1),
    "smoker": smoker,
    "region": region,
    "chronic_conditions": chronic_conditions,
    "cost": cost.round(2),
})
df.head()

In [ ]:
print(df.shape)
print(df["cost"].describe().round(0))

Claim costs are right-skewed: most are moderate, and a few are very large. That's typical for insurance data, and it affects how we evaluate models.

## 2. Split into training and test sets

The most important rule in machine learning is that a model must be judged on data it didn't learn from. We split the data before doing anything else. The test set is set aside and used only for the final check.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="cost")
y = df["cost"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train rows:", len(X_train))
print("test rows: ", len(X_test))

`X` is the feature table (everything except the target), and `y` is the target we want to predict. Capital `X` and lowercase `y` are the usual conventions.

## 3. Preprocessing with pipelines

Models need numbers, so text columns like `smoker` and `region` must be encoded. Numeric columns often benefit from scaling.

scikit-learn handles this with two tools:

- **`ColumnTransformer`** applies different steps to different columns.
- **`Pipeline`** chains preprocessing and a model into one object, so the same steps are applied to training and test data automatically.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ["age", "bmi", "chronic_conditions"]
categorical_features = ["smoker", "region"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# See what the preprocessing does on its own
preprocess.fit(X_train)
print(preprocess.get_feature_names_out())
print(preprocess.transform(X_train[:2]).round(2))

`StandardScaler` rescales numbers to mean 0 and standard deviation 1. `OneHotEncoder` turns each category into its own 0/1 column. `handle_unknown="ignore"` prevents an error if a new category shows up later.

## 4. Start with a baseline

Before trying clever models, compare against the simplest possible guess. A **dummy regressor** always predicts the average cost. If a real model can't beat it, something is wrong.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

baseline = make_pipeline(preprocess, DummyRegressor(strategy="mean"))
baseline.fit(X_train, y_train)

pred = baseline.predict(X_test)
print("baseline MAE:", round(mean_absolute_error(y_test, pred), 2))

`make_pipeline` is a shorter way to build a `Pipeline`. The steps are named automatically.

## 5. Linear regression

A linear model is fast, simple, and easy to explain. Each feature gets a weight, and the prediction is a weighted sum.

In [ ]:
from sklearn.linear_model import LinearRegression

linear = make_pipeline(preprocess, LinearRegression())
linear.fit(X_train, y_train)

pred_linear = linear.predict(X_test)
print("linear MAE:", round(mean_absolute_error(y_test, pred_linear), 2))

## 6. Gradient boosting

Gradient boosting (from the previous discussion) builds many small trees, each correcting the errors of the ones before it. It can capture non-linear effects and interactions that a straight line misses.

scikit-learn's `HistGradientBoostingRegressor` is fast, and it works well on tables of a few thousand rows or more. The preprocessing for trees doesn't need scaling, but the one-hot encoding is still needed, so we keep the same `preprocess` step.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

boosting = make_pipeline(
    preprocess,
    HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
    ),
)
boosting.fit(X_train, y_train)

pred_boost = boosting.predict(X_test)
print("boosting MAE:", round(mean_absolute_error(y_test, pred_boost), 2))

Key settings:

- **`max_iter`**: how many trees to build.
- **`learning_rate`**: how much each tree contributes. Smaller values need more trees.
- **`max_depth`**: how complex each tree can be. Deeper trees fit more and risk overfitting.

## 7. Evaluate properly

A single number can mislead, so use several metrics together:

- **MAE** (mean absolute error): the average size of a miss, in the same units as the target. Easy to explain.
- **RMSE** (root mean squared error): like MAE, but it punishes large misses more. Useful when big errors are costly.
- **R²**: the share of variation the model explains. 1.0 is perfect, and 0 means no better than the mean.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:<18} MAE={mae:8.1f}  RMSE={rmse:8.1f}  R2={r2:.3f}")

report("baseline", y_test, baseline.predict(X_test))
report("linear", y_test, pred_linear)
report("gradient boosting", y_test, pred_boost)

A single test split can be lucky or unlucky. **Cross-validation** splits the training data into several folds, trains on all but one, tests on the held-out fold, and repeats. The spread of scores tells you how stable the model is.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [("linear", linear), ("gradient boosting", boosting)]:
    scores = cross_val_score(model, X_train, y_train, cv=cv,
                             scoring="neg_mean_absolute_error")
    mae_scores = -scores                     # scikit-learn maximizes, so MAE is negated
    print(f"{name:<18} MAE per fold: {mae_scores.round(1)}  mean={mae_scores.mean():.1f}")

Notice that `cross_val_score` takes the whole pipeline. That's deliberate: preprocessing is fitted inside each fold, so no information leaks from the held-out data.

## 8. Which features matter?

**Permutation importance** shuffles one feature at a time and measures how much the score drops. A big drop means the model relies on that feature.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    boosting, X_test, y_test,
    n_repeats=10, random_state=42,
    scoring="neg_mean_absolute_error",
)

importance = pd.Series(result.importances_mean, index=X_test.columns)
importance.sort_values(ascending=False).round(1)

We built the data with the largest effects coming from `smoker` and `chronic_conditions`. Check whether the importances agree.

## 9. Saving and reloading a model

Because the whole pipeline is one object, you can save it and reuse it with `joblib`, which ships with scikit-learn:

In [ ]:
import joblib
from pathlib import Path
import tempfile

model_path = Path(tempfile.gettempdir()) / "claims_boosting.joblib"
joblib.dump(boosting, model_path)

reloaded = joblib.load(model_path)
print("predictions match:", np.allclose(reloaded.predict(X_test), pred_boost))

Only load model files you trust, since they can run code when loaded.

## 10. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** Change `max_depth` in the gradient boosting model to 2, then to 8. Which gives a lower test MAE? Explain why the deeper model is not always better.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
for depth in [2, 4, 8]:
    model = make_pipeline(
        preprocess,
        HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                      max_depth=depth, random_state=42),
    )
    model.fit(X_train, y_train)
    print(depth, round(mean_absolute_error(y_test, model.predict(X_test)), 1))
```

Deeper trees can fit noise in the training data, which may lower training error while raising test error. Compare the numbers to see the pattern on your run.

</details>

**Exercise 2.** Predict the *log* of the cost instead of the raw cost. Use `TransformedTargetRegressor` with `np.log1p` and `np.expm1`. Does the MAE change?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
from sklearn.compose import TransformedTargetRegressor

log_model = TransformedTargetRegressor(
    regressor=boosting,
    func=np.log1p,
    inverse_func=np.expm1,
)
log_model.fit(X_train, y_train)
report("log target", y_test, log_model.predict(X_test))
```

Modeling the log tames the skewed target. Whether the MAE improves depends on the data, so compare the numbers.

</details>

**Exercise 3.** Remove `region` from the features and retrain the gradient boosting model. How much does the test MAE change? What does that tell you about `region`?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
X_train_no_region = X_train.drop(columns="region")
X_test_no_region = X_test.drop(columns="region")

preprocess_no_region = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["smoker"]),
])

model = make_pipeline(
    preprocess_no_region,
    HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                  max_depth=4, random_state=42),
)
model.fit(X_train_no_region, y_train)
print(round(mean_absolute_error(y_test, model.predict(X_test_no_region)), 1))
```

Since the data was built without a regional effect, you should see little change. Real data sometimes shows a difference, so measure before you conclude.

</details>

## Summary

- **Split first**, and keep the test set for the final check.
- **Pipelines** bundle preprocessing and models, so steps are applied consistently.
- **Start with a baseline** so you know a model is actually helping.
- **Use several metrics** (MAE, RMSE, R²) and cross-validation for stable estimates.
- **Permutation importance** shows which features the model relies on.
- **Save whole pipelines** with `joblib`, not just the model.